# 06 · Silver learning
`bronze.learning_activity` → `silver.hcmlearningactivity` (module 06 §5; our addition to the design doc).

✅ Expect **17,923** rows · SUM(Hours) 90,585.60 · 13,999 completed · 8,336 mandatory.

In [ ]:
from pyspark.sql import functions as F
from hcm_utils import load_config, parse_date, latest_batch, issue_rows, write_issues, upsert, quarantine

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("run_id", "manual")
catalog = dbutils.widgets.get("catalog")
run_id  = dbutils.widgets.get("run_id")
cfg = load_config()

In [ ]:
SRC_FILE, REF = "LEARNING_ACTIVITY", "ENROLLMENT_ID"
COLS = ["PERSON_ID", "ENROLLMENT_ID", "COURSE_ID", "COURSE_NAME", "COURSE_CATEGORY", "ENROLLMENT_DATE",
        "COMPLETION_STATUS_CODE", "COMPLETION_DATE", "HOURS", "MANDATORY_FLAG", "CERTIFICATE_ID"]
hours_type = cfg["learning"]["hours_type"]

df = (latest_batch(f"{catalog}.bronze.learning_activity").select(*COLS)
      .withColumn("ENR_DATE", parse_date("ENROLLMENT_DATE"))
      .withColumn("CMP_DATE", parse_date("COMPLETION_DATE"))
      .withColumn("HRS", F.expr(f"try_cast(HOURS AS {hours_type})")))
assert df.filter("ENR_DATE IS NULL").count() == 0, "unparseable ENROLLMENT_DATE - add a format to parse_date()"

issues = [
    issue_rows(df.filter(~F.col("ENROLLMENT_DATE").rlike(r"^\d{4}-\d{2}-\d{2}$")), SRC_FILE, REF,
               "FORMAT_INCONSISTENCY", "ENROLLMENT_DATE in non-ISO format", "FIXED"),
    issue_rows(df.filter(~F.col("HOURS").rlike(r"^-?\d+\.\d$")), SRC_FILE, REF,
               "FORMAT_INCONSISTENCY", "HOURS stored as inconsistent string precision", "FIXED"),
]
bad = df.filter(F.col("HRS") < cfg["learning"]["min_hours"])
issues.append(issue_rows(bad, SRC_FILE, REF, "INVALID_VALUE", "HOURS is negative", "QUARANTINED"))
quarantine(bad.select(*COLS), SRC_FILE, REF, "HOURS is negative", catalog, run_id)

# try_cast turns non-numeric HOURS into NULL; never let those rows vanish silently
unparseable = df.filter(F.col("HOURS").isNotNull() & F.col("HRS").isNull())
issues.append(issue_rows(unparseable, SRC_FILE, REF, "UNPARSEABLE_VALUE", "HOURS is not numeric", "QUARANTINED"))
quarantine(unparseable.select(*COLS), SRC_FILE, REF, "HOURS is not numeric", catalog, run_id)  # 0 rows in this data

In [ ]:
silver = (df.filter(F.col("HRS") >= cfg["learning"]["min_hours"])
    .select(
        F.col("ENROLLMENT_ID").cast("bigint").alias("EnrollmentId"),
        F.col("PERSON_ID").cast("bigint").alias("PersonId"),
        F.trim("COURSE_ID").alias("CourseId"),
        F.trim("COURSE_NAME").alias("CourseName"),
        F.trim("COURSE_CATEGORY").alias("CourseCategory"),
        F.col("ENR_DATE").alias("EnrollmentDate"),
        F.upper(F.trim("COMPLETION_STATUS_CODE")).alias("CompletionStatus"),
        F.col("CMP_DATE").alias("CompletionDate"),
        F.col("HRS").alias("Hours"),
        (F.upper(F.trim("MANDATORY_FLAG")) == "Y").alias("IsMandatory"),
        (F.upper(F.trim("COMPLETION_STATUS_CODE")) == "COMPLETE").alias("IsCompleted"),
        F.trim("CERTIFICATE_ID").alias("CertificateId"),
        F.lit(run_id).alias("RunId"),
        F.current_timestamp().alias("SilverLoadTimestamp")))
upsert(silver, f"{catalog}.silver.hcmlearningactivity", keys=["EnrollmentId"])
print("issues logged:", write_issues(issues, catalog, run_id))
print("hcmlearningactivity rows:", spark.table(f"{catalog}.silver.hcmlearningactivity").count())